# Published SNP-heritability against `h2_Unrel` — 1kg_eur

Puts our per-phenotype `h2_Unrel` beside published SNP-heritability for the
same traits in European-ancestry samples from All of Us and UK Biobank, with
the related-pair estimators in the same row order on the right. Reads
`09_estimators.ipynb`'s `estimators_<run>.tsv` and the literature table
`docs/published_h2.tsv`. It computes no new estimates, only the pooled
`h2_Unrel` for the dashed reference line (the same random-effects pooling as
`12_meta_analysis.ipynb`).

**The literature table.** One row per (phenotype, source, estimator), with the
citation, DOI or URL, and where in the source the number is. Every value was
read from the source's own supplementary table or public manifest, not
transcribed by hand:

| cohort | source | estimator | traits |
|---|---|---|---|
| AoU EUR | Tsuo et al., *Nat Genet* 2026, Supp. Table 2 | LDSC; GCTA-GREML on downsampled unrelated | 11 |
| AoU EUR | Cole et al., bioRxiv 2026 (preprint), Table S2 | LDSC | 6 |
| UKB EUR | Tsuo et al. 2026, same table | GCTA-GREML, matched N | 11 |
| UKB EUR | Pan-UKB h2 manifest (Karczewski et al., *Nat Genet* 2025) | LDSC, S-LDSC 25-bin | 31 |
| UKB EUR | Pan-UKB h2 manifest | RHE-mc (EUR pilot traits only) | 11 |
| UKB white British | Neale lab round 2 | S-LDSC, Baseline-LD v1.1 | 30 |

**The samples differ, and it matters for reading the figure.**
- *Ancestry.* Tsuo and Pan-UKB "EUR" is a random forest on 1000 Genomes + HGDP
  PCs (includes non-British Europeans). Cole's AoU set is narrower —
  "White British-like", matched to UKB in GBMI PC space. Neale is UKB white
  British. Our cohort is CEU-anchored, so closest in definition to Cole and Neale.
- *Relatives.* Tsuo's AoU LDSC and Pan-UKB's LDSC/S-LDSC come from mixed-model
  GWAS with relatives kept; the GREML, RHE-mc, Cole and Neale estimates are on
  unrelated people only. `h2_Unrel` keeps everyone but fits only pairs with
  `a_ij < 0.02`, so it is closest to the unrelated-only designs.
- *Sample size.* Tsuo's GREML samples were downsampled to the smallest unrelated
  training set for each trait (N not reported; at most ~13–43K), hence their
  wider SEs.

**What is drawn.** Only rows with `match` = `exact` or `close` (same measurement;
`close` means a transform or definition differs, recorded in `note`). The
`proxy` rows — a single UKB question standing in for a multi-item AoU scale
(loneliness, social support, AUDIT-C) — are in the table, not on the figure. PSS
and EDS have no UKB counterpart, and most lipids, glycaemic traits, several
blood counts and every behavioural score have no published AoU EUR estimate.

**How to read it.** The GRM-based markers (GREML, RHE-mc) are the like-for-like
comparison, since `h2_Unrel` is Haseman–Elston regression on a GRM too.
Summary-statistic estimates run lower: on the 11 traits where Pan-UKB ran both,
its RHE-mc is above its univariate LDSC every time.

## Setup

In [ ]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"

REPO_DIR = os.environ.get("AOUCOV_REPO") or next(
    (q for q in (os.path.expanduser("~/repos/aou-covariance"),
                 os.path.expanduser("~/repos/AOU-covariance"),
                 os.path.expanduser("~/aou-covariance"))
     if os.path.isdir(os.path.join(q, "src"))), None)
assert REPO_DIR, "cannot find the repo checkout -- set AOUCOV_REPO"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
EST_DIR = (f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}"
           f"/07_estimators/he_regression")
OUT_DIR = (f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}"
           f"/07_estimators/published_h2")
os.makedirs(f"{OUT_DIR}/plots", exist_ok=True)

EST_PATH = f"{EST_DIR}/estimators_{RUN_DIR}.tsv"
assert os.path.isfile(EST_PATH), (
    f"missing {EST_PATH} — run 09_estimators.ipynb first"
)
E = pd.read_csv(EST_PATH, sep="\t")

# 09_estimators.ipynb writes ci_lo/ci_hi/ci_level per row (t on ci_df). Take the
# level from the file rather than restating it, so the two cannot disagree.
CI_LEVEL = (float(E["ci_level"].dropna().iloc[0]) if "ci_level" in E.columns
            and E["ci_level"].notna().any() else 0.95)
HAVE_CI = {"ci_lo", "ci_hi"} <= set(E.columns)
assert HAVE_CI, (
    f"{EST_PATH} has no ci_lo/ci_hi — re-run 09_estimators.ipynb, which now "
    f"exports the interval alongside the SE")

# ── shared style, palettes and helpers ────────────────────────────────────────
sys.path.insert(0, f"{REPO_DIR}/src")
import importlib

import covplot as CP
import he_estimators as HE
importlib.reload(CP)
importlib.reload(HE)

# A stale covplot on the VM used to surface as an AttributeError deep inside a
# drawing cell, or -- worse -- as a NameError later still, because the setup
# cell had already failed and been scrolled past. Check here, by name, so the
# message says what to do.
_missing = [_n for _n in ['apply_style', 'collect_run_plots', 'mirror_plots', 'repel_labels'] if not hasattr(CP, _n)]
assert not _missing, (
    f"src/covplot.py is out of date: {', '.join(_missing)} missing.\n"
    f"  git -C {REPO_DIR} pull      then restart the kernel if it persists")
# and the API level, which a name check cannot see: a function whose SIGNATURE
# changed is still present under its own name, and fails much further in.
assert getattr(CP, "API", 0) >= 1, (
    f"src/covplot.py is API {getattr(CP, 'API', 0)}, this notebook needs >= 1.\n"
    f"  git -C {REPO_DIR} pull      then restart the kernel if it persists")

SURFACE, INK, INK2, MUTED, GRID, AXIS = (CP.SURFACE, CP.INK, CP.INK2, CP.MUTED,
                                         CP.GRID, CP.AXIS)
CLASS_COL, RAMP = CP.CLASS_COL, CP.RAMP
CATEGORY, CAT_ORDER, CAT_COL = CP.CATEGORY, CP.CAT_ORDER, CP.CAT_COL
FIG_TRANSPARENT = False        # white background; True for transparent
FONT = CP.apply_style(transparent=FIG_TRANSPARENT)

TRANSFORMS = sorted(E["transform"].unique())
COVSETS_ALL = sorted(E["covset"].unique())
COVSETS = [c for c in ("base", "base_pcs", "base_pcs_zip3", "base_pcs_zip3_ses")
           if c in COVSETS_ALL] or COVSETS_ALL
REF_TF = "invnorm" if "invnorm" in TRANSFORMS else TRANSFORMS[0]
REF_CS = "base_pcs" if "base_pcs" in COVSETS else COVSETS[-1]

print(f"{len(E):,} estimator rows | {E['phenotype'].nunique()} phenotypes")
print(f"transforms: {TRANSFORMS}\ncovsets:    {COVSETS}")
print(f"reference:  {REF_TF} / {REF_CS}")
print(f"intervals:  {100*CI_LEVEL:.0f}%  (per-phenotype: t on "
      f"{int(E['ci_df'].dropna().iloc[0])} df; pooled: normal)")
print(f"out:        {OUT_DIR}")

# CATEGORY / CAT_ORDER / CAT_COL come from covplot, as in 12_meta_analysis
E["category"] = E["phenotype"].map(CATEGORY).fillna("uncategorised")

## Pooled `h2_Unrel` — the dashed line

DerSimonian–Laird random effects over phenotypes, the same arithmetic as
`12_meta_analysis.ipynb`'s `pool()`, so the line sits where that notebook's
Figures 1–3 put it.

In [ ]:
def pool_re(est, se):
    """DerSimonian-Laird random-effects mean of `est`, as in 12's `pool()`."""
    est, se = np.asarray(est, float), np.asarray(se, float)
    ok = np.isfinite(est) & np.isfinite(se) & (se > 0)
    est, se = est[ok], se[ok]
    k = len(est)
    if k == 0:
        return dict(k=0, re=np.nan)
    w = 1.0 / se ** 2
    fe = (w * est).sum() / w.sum()
    Q = (w * (est - fe) ** 2).sum()
    denom = w.sum() - (w ** 2).sum() / w.sum()
    tau2 = max(0.0, (Q - (k - 1)) / denom) if k > 1 and denom > 0 else 0.0
    ws = 1.0 / (se ** 2 + tau2)
    return dict(k=k, re=float((ws * est).sum() / ws.sum()))


_u = E[(E["estimator"] == "h2_Unrel") & (E["classes"] == "noPO")
       & (E["transform"] == REF_TF) & (E["covset"] == REF_CS)]
overall = pd.DataFrame([pool_re(_u["est"], _u["se"])])
print(f"pooled h2_Unrel ({REF_TF}, {REF_CS}): {overall.iloc[0]['re']:.4f} "
      f"over k={int(overall.iloc[0]['k'])} phenotypes")

## The literature table

Loaded from the repo, so it arrives with `git pull`. The comparison printed
below — ours beside every drawn source, one row per phenotype — is the figure's
left panel as numbers, and is written next to the PNG.

In [ ]:
LIT_PATH = f"{REPO_DIR}/docs/published_h2.tsv"
assert os.path.isfile(LIT_PATH), (
    f"missing {LIT_PATH} -- git -C {REPO_DIR} pull")
LIT = pd.read_csv(LIT_PATH, sep="\t")
_unk = sorted(set(LIT["phenotype"]) - set(E["phenotype"]))
if _unk:
    print(f"in the table but not in this run: {_unk}")
print(LIT.groupby(["cohort", "label", "match"]).size().unstack(fill_value=0).to_string())

_ours = _u.set_index("phenotype")["est"].rename("this study: h2_Unrel")
_drawn = LIT[LIT["match"].isin(["exact", "close"])]
CMP = (_drawn.pivot_table(index="phenotype", columns="label", values="h2")
       .reindex(columns=list(dict.fromkeys(_drawn["label"]))))
CMP = pd.concat([_ours, CMP], axis=1).loc[lambda d: d.iloc[:, 0].notna()]
CMP = CMP.loc[CMP.iloc[:, 0].sort_values(ascending=False).index]
CMP.to_csv(f"{OUT_DIR}/published_h2_comparison_{RUN_DIR}.tsv", sep="\t")
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(CMP.round(3).to_string())

## Figure — every phenotype against the literature

Left: our `h2_Unrel` (black, with its CI) and the published estimates for the
same trait on the same row line — All of Us in red, UKB in violet, shape for the
method: diamond LDSC, circle GREML, square RHE-mc, triangles S-LDSC. Hollow and
drawn underneath, so a source that agrees with us rings our point. Right: the
related-range estimators for the same phenotype in the same row, including all
three of `h2_RelOffsets`' per-band slopes. Rows are grouped by trait category
(tinted) and ordered by `h2_Unrel` within each; the dashed line is the pooled
`h2_Unrel`. Sized for a slide.

In [ ]:
# `h2_RelOffsets` is not a fourth model to plot alongside the bands: the module
# reports the TOP band's slope as its headline (`b[NB-1]`), so a point for it
# is literally the band3 point drawn twice. The three per-band slopes go in
# instead -- that model fits a separate slope in each band, and one marker
# could never say which. Shades match 11_crossproduct_curves.ipynb, so a
# marker here and a line there are the same colour.
_bands = list(HE.OFFSET_BANDS)
REL_MODELS = (
    [("h2_Rel", CP.MODEL_COL["h2_Rel"], "o", "related: slope + int"),
     ("h2_OneSlopeOffsets", CP.MODEL_COL["h2_OneSlopeOffsets"], "s",
      "1 slope, band ints")]
    + [(f"RelOffsets.slope.{d}", col, "^",
        f"band slope {HE.OFFSET_BANDS[d][0]:g}-{HE.OFFSET_BANDS[d][1]:g}")
       for d, col in zip(_bands, CP.shades(CP.MODEL_COL["h2_RelOffsets"],
                                           len(_bands)))])

sel = E[(E["transform"] == REF_TF) & (E["covset"] == REF_CS)
        & (E["classes"] == "noPO")]
IDX = ["category", "phenotype"]
# dropna=False, then reindex the SE frame onto the estimate frame: pivot_table
# drops any column that is entirely NaN, so an estimator whose jackknife SE
# failed everywhere would otherwise vanish from wide_se alone and the two
# frames would stop lining up.
wide = sel.pivot_table(index=IDX, columns="estimator", values="est", dropna=False)
wide = wide[wide["h2_Unrel"].notna()]
def _pv(col):
    return (sel.pivot_table(index=IDX, columns="estimator", values=col, dropna=False)
            .reindex(index=wide.index, columns=wide.columns))


# the per-phenotype bars are he_estimators' t intervals, read off the export
wide_lo, wide_hi = _pv("ci_lo"), _pv("ci_hi")

# group by category, then by h2_Unrel within it
order = []
for cat in [c for c in CAT_ORDER if c in wide.index.get_level_values(0)]:
    blk = wide.loc[cat].sort_values("h2_Unrel", ascending=False)
    order += [(cat, p) for p in blk.index]
wide = wide.reindex(order)
wide_lo, wide_hi = wide_lo.reindex(order), wide_hi.reindex(order)


def _xerr(est):
    return [wide[est] - wide_lo[est], wide_hi[est] - wide[est]]

y = np.arange(len(order))

# a strip under both panels for the literature legend: at slide size it is
# wider than the left panel, so inside that panel it covered markers, and as
# an axes legend hung below it constrained_layout narrowed the panel to fit
_main_h, _leg_h = 0.21 * len(order) + 2.0, 0.8
fig = plt.figure(figsize=(14.5, _main_h + _leg_h), constrained_layout=True)
_gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.5], height_ratios=[_main_h, _leg_h])
axL = fig.add_subplot(_gs[0, 0])
axR = fig.add_subplot(_gs[0, 1], sharey=axL)
axLeg = fig.add_subplot(_gs[1, :])
axLeg.axis("off")

axL.errorbar(wide["h2_Unrel"], y, xerr=_xerr("h2_Unrel"), fmt="o", ms=5.5,
             lw=1.2, color=INK, zorder=3, label="this study: $h^2_{\\mathrm{Unrel}}$")

# Published SNP-heritability for the same trait (docs/published_h2.tsv, one row
# per source with its citation). One hollow marker per source, no interval --
# their SEs are ~0.01, and the point is where the literature sits, not a test.
# All on the row's own line, not dodged: offset markers read as belonging to
# the neighbouring row. Hollow and a size up from ours, drawn underneath, so
# where a source agrees with us it rings our point instead of hiding it.
# Colour is the cohort, shape the method. Only rows whose
# trait matches ours are drawn: the `proxy` rows (a one-item UKB question
# standing in for a multi-item scale) stay in the table, off the figure.
lit = LIT[LIT["match"].isin(["exact", "close"])]
_row = {p: i for i, (_, p) in enumerate(order)}
LIT_COL = {"AoU": "#e34948", "UKB": "#4a3aa7"}
# shape is the METHOD, the same in both cohorts: diamond LDSC, circle GREML,
# square RHE-mc, triangles S-LDSC. A source not listed here gets an X, which
# is a cue to add it rather than a shape that means anything.
LIT_MARK = {"tsuo_ldsc": "D", "cole_ldsc": "P", "panukb_ldsc": "D",
            "tsuo_greml": "o", "panukb_rhemc": "s",
            "panukb_sldsc": "v", "neale_sldsc": "^"}
for _c, _col in LIT_COL.items():
    _sub = lit[(lit["cohort"] == _c) & lit["phenotype"].isin(_row)]
    for _src in _sub["source"].unique():
        _s, _mk = _sub[_sub["source"] == _src], LIT_MARK.get(_src, "X")
        axL.plot(_s["h2"], _s["phenotype"].map(_row), _mk, ms=6.2,
                 mfc="none", mec=_col, mew=1.0, ls="none", zorder=2.5,
                 label=_s["label"].iloc[0])
if len(overall):
    for ax_ in (axL, axR):
        ax_.axvline(overall.iloc[0]["re"], color=INK2, lw=0.8, ls=(0, (4, 2)),
                    zorder=1)
axL.set_xlabel(f"$h^2_{{\\mathrm{{Unrel}}}}$, {100*CI_LEVEL:.0f}% CI")
axL.set_title("unrelated pairs", loc="left", fontsize=13)
# ours first: the errorbar is drawn before the literature but legend() lists
# errorbar containers after plain lines
_h, _l = axL.get_legend_handles_labels()
_k = sorted(range(len(_l)), key=lambda i: not _l[i].startswith("this study"))
axLeg.legend([_h[i] for i in _k], [_l[i] for i in _k], fontsize=9.5, ncol=5,
             loc="upper left", frameon=False, columnspacing=1.4,
             handletextpad=0.3, labelspacing=0.4,
             title="SNP-$h^2$ in European-ancestry samples", title_fontsize=10)

# The band intervals are wide enough to set this axis on their own and flatten
# every other series onto the zero line. Same rule as 09's forest: a fixed
# window, widened only by a point estimate, with arrowheads where an interval
# runs off. The LEFT panel stays data-driven -- its job is fine resolution of
# h2_Unrel over roughly 0.05-0.7, which a shared window would compress.
_shown = [m for m in REL_MODELS if m[0] in wide.columns]
_rlim = CP.clip_limits(np.concatenate([wide[e].to_numpy(float) for e, *_ in _shown]),
                       (-0.5, 1.5)) if _shown else None
if _rlim:
    axR.set_xlim(*_rlim)
for k, (est, col, mk, lab) in enumerate(REL_MODELS):
    if est not in wide.columns:
        continue
    # five series per row, dodged by 0.07 so the whole group spans +-0.14 --
    # at 0.15 the outer ones sat nearer the next row's than their own
    dy = (k - (len(REL_MODELS) - 1) / 2) * 0.07
    # err_alpha, not alpha: matplotlib's alpha would fade the markers with
    # the bars. Here the points are the reading and the intervals are context.
    CP.capped_errorbar(axR, wide[est].to_numpy(float), y + dy,
                       wide_lo[est].to_numpy(float), wide_hi[est].to_numpy(float),
                       fmt=mk, ms=5, lw=0.9, color=col, err_alpha=0.30,
                       elinewidth=0.9, label=lab, cap_ms=5, out_ms=8, zorder=3)
axR.set_xlabel(f"estimate, {100*CI_LEVEL:.0f}% CI")
axR.set_title("related pairs — same row order", loc="left", fontsize=13)
axR.legend(fontsize=9.5, loc="lower right", labelspacing=0.35,
           handletextpad=0.5, borderpad=0.3)

axL.set_yticks(y)
axL.set_yticklabels([p for _, p in order], fontsize=10.5)
# sized for a slide: the rcParams defaults (9 pt) are for the notebook
for ax_ in (axL, axR):
    ax_.tick_params(axis="x", labelsize=10.5)
    ax_.xaxis.label.set_size(12)
# rows fill the panel exactly, top row first, so the category bands below
# run edge to edge instead of stopping short of an autoscale margin
axL.set_ylim(len(order) - 0.5, -0.5)
axR.tick_params(axis="y", labelleft=False)   # shared rows; names on the left only
# category separators, each block on a faint tint of its category colour --
# behind the grid, so it groups rows without competing with the markers -- and
# named just outside the right panel, where no marker or legend can reach it
seen = []
for i_, (cat, _) in enumerate(order):
    if cat not in seen:
        _n = sum(1 for c_, _ in order if c_ == cat)
        for ax_ in (axL, axR):
            ax_.axhspan(i_ - 0.5, i_ + _n - 0.5, color=CAT_COL.get(cat, MUTED),
                        alpha=0.07, lw=0, zorder=0)
        if i_:
            for ax_ in (axL, axR):
                ax_.axhline(i_ - 0.5, color=GRID, lw=0.8)
        axR.annotate(cat, (1.0, i_ + (_n - 1) / 2), xycoords=("axes fraction", "data"),
                     ha="left", va="center", fontsize=10.5, color=MUTED,
                     xytext=(4, 0), textcoords="offset points", annotation_clip=False)
        seen.append(cat)

fig.suptitle(f"Per-phenotype estimates against published SNP-heritability — "
             f"{REF_TF}, {REF_CS}", fontsize=14)
png = f"{OUT_DIR}/plots/published_h2_vs_unrel_related.png"
fig.savefig(png, dpi=220, bbox_inches="tight")
plt.show()
print(f"-> {png}")
print(f"{len(order)} phenotypes ordered within "
      f"{len(seen)} categor{'y' if len(seen)==1 else 'ies'}")

In [ ]:
CP.mirror_plots(f"{OUT_DIR}/plots", "published_h2", RUN_DIR)

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser("~/repos/AOU-covariance/notebooks/12a_published_h2.ipynb")
_gs = ("gs://cloned-shared-env-pilot-wb-swift-sprout-7231/"
       f"{PROJECT_DIR}/{RUN_DIR}/notebooks/12a_published_h2.ipynb")
subprocess.run(["gcloud", "storage", "cp", _nb, _gs], check=True)
print(f"notebook -> {_gs}")